In [16]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

2.13.0+cu130
True
NVIDIA GB10


In [45]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, pipeline, BertTokenizerFast, BertForTokenClassification, BertForQuestionAnswering, get_linear_schedule_with_warmup
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.optim import AdamW

| What you imported | What it does | Everyday analogy |
|---|---|---|
| `BertTokenizer` | Turns text into numbers the model can read | A translator — "great movie" → `[101, 2307, 3185, 102]` |
| `BertForSequenceClassification` | The BERT model, set up to **label whole sentences** (e.g. positive/negative) | The brain |
| `TrainingArguments` | Your settings: learning rate, batch size, how many epochs | The dials on a machine |
| `Trainer` | Runs the actual training loop for you | The mechanic who operates the machine |

In [3]:
dataset = load_dataset("imdb")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 511686.44 examples/s]


In [17]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset  = dataset["test"].shuffle(seed=42).select(range(5000))

In [18]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/home/ankitanand/Documents/pp/Finetuning_HF/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


#### The raw text column (Each row in the IMDB dataset contains a review text)
It will pad each sentence at an exact length of 256 tokens
If the text is longer than 256 tokens, it will trauncate it.

In [19]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

In [20]:
# Apply tokenizaiton + rename + format in a single flow
def preprocess(ds):
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [21]:
train_dataset = preprocess(train_dataset)

Map: 100%|██████████| 10000/10000 [00:12<00:00, 770.47 examples/s]


In [22]:
test_dataset = preprocess(test_dataset)

Map: 100%|██████████| 5000/5000 [00:06<00:00, 788.60 examples/s]


In [15]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
training_args = TrainingArguments(
    output_dir="./bert-finetuned-imdb",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    logging_dir="./BERTlogs",
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True,
    logging_steps=50,
    report_to="none"
)

In [27]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [28]:
trainer.train()

Step,Training Loss
50,0.543700
100,0.355500
150,0.309800
200,0.303500
250,0.278600
300,0.263100
350,0.198900
400,0.157500
450,0.188400
500,0.157300


TrainOutput(global_step=626, training_loss=0.25614203888768206, metrics={'train_runtime': 143.0577, 'train_samples_per_second': 139.804, 'train_steps_per_second': 4.376, 'total_flos': 2631110553600000.0, 'train_loss': 0.25614203888768206, 'epoch': 2.0})

In [ ]:
# saving the model
trainer.save_model("./bert-finetuned-imdb")

In [30]:
# saving the tokenizer
tokenizer.save_pretrained("./bert-finetuned-imdb")

('./bert-finetuned-imdb/tokenizer_config.json',
 './bert-finetuned-imdb/special_tokens_map.json',
 './bert-finetuned-imdb/vocab.txt',
 './bert-finetuned-imdb/added_tokens.json')

In [31]:
metrics = trainer.evaluate()

In [33]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
trainer.evaluate()

{'eval_loss': 0.22927841544151306,
 'eval_model_preparation_time': 0.0012,
 'eval_accuracy': 0.915,
 'eval_f1': 0.9157247670037676,
 'eval_runtime': 7.8548,
 'eval_samples_per_second': 636.551,
 'eval_steps_per_second': 79.569}

### Prediction

In [34]:
tokenizer = BertTokenizer.from_pretrained("./bert-finetuned-imdb")
model = BertForSequenceClassification.from_pretrained("./bert-finetuned-imdb")

In [37]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

### Predict

In [41]:
text = "The plot was predictable but the performances were moving."
result = classifier(text)

In [42]:
result

[{'label': 'LABEL_1', 'score': 0.9754239320755005}]

### Pushing to Huggingfacehub

In [43]:
tokenizer.push_to_hub("ankitanand9/my-bert-imdb")

CommitInfo(commit_url='https://huggingface.co/ankitanand9/my-bert-imdb/commit/f0a417644493ed2edb5caa422f1f532cba101fea', commit_message='Upload tokenizer', commit_description='', oid='f0a417644493ed2edb5caa422f1f532cba101fea', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ankitanand9/my-bert-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='ankitanand9/my-bert-imdb'), pr_revision=None, pr_num=None)

In [44]:
trainer.push_to_hub("ankitanand9/my-bert-imdb")

Processing Files (2 / 2): 100%|██████████|  438MB /  438MB, 12.0MB/s  
New Data Upload: 100%|██████████|  430MB /  430MB, 12.0MB/s  


CommitInfo(commit_url='https://huggingface.co/ankitanand9/bert-finetuned-imdb/commit/fcdf4b9570c45f5838294d0a67f0ef800e7f9511', commit_message='ankitanand9/my-bert-imdb', commit_description='', oid='fcdf4b9570c45f5838294d0a67f0ef800e7f9511', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ankitanand9/bert-finetuned-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='ankitanand9/bert-finetuned-imdb'), pr_revision=None, pr_num=None)

## Finetune on multiple problem

In [46]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda
